This file performs the following  
Splits the adata into liver and immune collections (Split GSE192740 into immune and liver adata for this)  
For each collection:  
  - Find genes expressed in ≥10 cells per dataset
  - Take intersection across all datasets in collection
  - Concatenate on shared genes (inner join on filtered universe)
  - HVG selection with batch_key='dataset_id', n_top=5000
  - Batch correction (scVI or Harmony) preserving biology  

For each collection perform CellTypist liver model cell annotation  
Check cell counts and create per cell type adata objects  

In [1]:
from pathlib import Path

import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.stats import median_abs_deviation
import numpy as np

# Set Base directory to the location of this script
BASE = Path('c:/Users/ankit/Documents/scFM/train_data/')

# Set working directory to the location of this script 
# data_dir = BASE / 'h5s_common_directory'
data_dir = BASE / 'h5s_common_directory_V2_WIP'
# data_dir = BASE / 'scGPT_data'

# Print contents of the data directory for verification
print(f"Contents of data directory ({data_dir}):")
for item in data_dir.iterdir():
    print(item.name)

Contents of data directory (c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIP):
GSE159977.h5ad
GSE174748.h5ad
GSE185477.h5ad
GSE189600.h5ad
GSE190487.h5ad
GSE192740.h5ad
GSE202379.h5ad
GSE212837.h5ad
GSE270488.h5ad


In [2]:
# Read in each h5ad file with the Anndata object variable name being it's file name without the extension and store in dictionary
adata_dict = {}

for h5ad_file in data_dir.glob('*.h5ad'):
    adata_name = h5ad_file.stem  # Get file name without extension
    adata_dict[adata_name] = ad.read_h5ad(h5ad_file)  # Read h5ad file and store in dictionary

c:\Users\ankit\miniconda3\envs\celltypist\Lib\site-packages\anndata\_core\anndata.py:1878: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Sanity Check

In [3]:
for adata_name, adata in adata_dict.items():
    print(f"\nProcessing Anndata object: {adata_name}")
    print(f"Shape of the Anndata object: {adata.shape}")
    print(f"Number of genes: {adata.n_vars}")
    print(f"Number of cells: {adata.n_obs}")


Processing Anndata object: GSE159977
Shape of the Anndata object: (106851, 23993)
Number of genes: 23993
Number of cells: 106851

Processing Anndata object: GSE174748
Shape of the Anndata object: (19038, 27934)
Number of genes: 27934
Number of cells: 19038

Processing Anndata object: GSE185477
Shape of the Anndata object: (125790, 37469)
Number of genes: 37469
Number of cells: 125790

Processing Anndata object: GSE189600
Shape of the Anndata object: (55074, 32277)
Number of genes: 32277
Number of cells: 55074

Processing Anndata object: GSE190487
Shape of the Anndata object: (41995, 19059)
Number of genes: 19059
Number of cells: 41995

Processing Anndata object: GSE192740
Shape of the Anndata object: (69758, 30704)
Number of genes: 30704
Number of cells: 69758

Processing Anndata object: GSE202379
Shape of the Anndata object: (64329, 30596)
Number of genes: 30596
Number of cells: 64329

Processing Anndata object: GSE212837
Shape of the Anndata object: (182302, 33557)
Number of genes: 

In [10]:
print(adata_dict["GSE192740"].obs["Sample name"].value_counts())

# Assign the cells with Sample name "Liver_CD45- Cells_ Human" a assay_type value of scRNA-seq
adata_dict["GSE192740"].obs.loc[adata_dict["GSE192740"].obs["Sample name"] == "Liver_CD45- Cells_ Human", "assay_type"] = "scRNA-seq"

print(adata_dict["GSE192740"].obs["Sample name"].value_counts())


Sample name
Whole Liver Nuclei_Human    39020
Liver_CD45+ Cells_ Human    27830
Liver_CD45- Cells_ Human     2908
Name: count, dtype: int64
Sample name
Whole Liver Nuclei_Human    39020
Liver_CD45+ Cells_ Human    27830
Liver_CD45- Cells_ Human     2908
Name: count, dtype: int64


In [11]:
print(adata_dict["GSE192740"].obs["assay_type"].value_counts())

# Create new GSE192740_liver and GSE192740_immune by filtering GSE192740 for assay_type == 'liver' and assay_type == 'immune' respectively
adata_dict["GSE192740_liver"] = adata_dict["GSE192740"][adata_dict["GSE192740"].obs["assay_type"] == "snRNA-seq"].copy()
adata_dict["GSE192740_immune"] = adata_dict["GSE192740"][adata_dict["GSE192740"].obs["assay_type"] == "scRNA-seq"].copy()

# Print the Sample name value counts for the new GSE192740_liver and GSE192740_immune Anndata objects to verify the filtering
print(adata_dict["GSE192740_liver"].obs["Sample name"].value_counts())
print(adata_dict["GSE192740_immune"].obs["Sample name"].value_counts())

assay_type
snRNA-seq    39020
scRNA-seq    30738
None             0
Name: count, dtype: int64
Sample name
Whole Liver Nuclei_Human    39020
Name: count, dtype: int64
Sample name
Liver_CD45+ Cells_ Human    27830
Liver_CD45- Cells_ Human     2908
Name: count, dtype: int64


In [13]:
# See what is the number of unique genes that are present in all adata objects
unique_genes = set()
for adata_name, adata in adata_dict.items():
    unique_genes.update(adata.var_names)

print(f"Number of unique genes in all adata objects: {len(unique_genes)}")


Number of unique genes in all adata objects: 48422


In [14]:
# See what is the number of genes that are present in all adata objects
common_genes = set(adata_dict["GSE192740"].var_names)
for adata_name, adata in adata_dict.items():
    common_genes.intersection_update(adata.var_names)

print(f"Number of common genes in all adata objects: {len(common_genes)}")

Number of common genes in all adata objects: 12744


In [15]:
# Get rid of the GSE192740 Anndata object from the adata_dict since we have created GSE192740_liver and GSE192740_immune from it
del adata_dict["GSE192740"]



Check for duplicated cells

In [19]:
for name, adata in adata_dict.items():
    print(name)
    print(f"Duplicate cells before: {adata.obs.index.duplicated().sum()}")
    
    # Remove duplicate cells if they exist
    if adata.obs.index.duplicated().any():
        adata = adata[~adata.obs.index.duplicated(keep='first')]
        adata_dict[name] = adata
    
    print(f"Duplicate cells after: {adata.obs.index.duplicated().sum()}")


GSE159977
Duplicate cells before: 3609
Duplicate cells after: 0
GSE174748
Duplicate cells before: 0
Duplicate cells after: 0
GSE185477
Duplicate cells before: 0
Duplicate cells after: 0
GSE189600
Duplicate cells before: 0
Duplicate cells after: 0
GSE190487
Duplicate cells before: 0
Duplicate cells after: 0
GSE202379
Duplicate cells before: 0
Duplicate cells after: 0
GSE212837
Duplicate cells before: 0
Duplicate cells after: 0
GSE270488
Duplicate cells before: 0
Duplicate cells after: 0
GSE192740_liver
Duplicate cells before: 0
Duplicate cells after: 0
GSE192740_immune
Duplicate cells before: 0
Duplicate cells after: 0


Perform annotation

In [2]:
from pathlib import Path

import celltypist
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.stats import median_abs_deviation
import numpy as np
import celltypist
from celltypist import models

c:\Users\ankit\miniconda3\envs\celltypist\Lib\site-packages\celltypist\classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [18]:
model = models.Model.load(model = 'Healthy_Human_Liver.pkl')

In [20]:
for name, adata in adata_dict.items():
    print(name)
    
    # Skip if already annotated
    if any(col.startswith("typist_liver_") for col in adata.obs.columns):
        print("Already annotated with typist_liver_, skipping...")
        print()
        continue
    
    print()
    predictions = celltypist.annotate(adata, model = 'Healthy_Human_Liver.pkl', majority_voting = True)
    adata = predictions.to_adata(prefix="typist_liver_", insert_conf_by="majority_voting")
    adata_dict[name] = adata
    print(adata.obs["typist_liver_majority_voting"].head())
    print(adata.obs["typist_liver_conf_score"].head())

GSE159977



🔬 Input data has 103242 cells and 23993 genes
🔗 Matching reference genes in the model
🧬 2568 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
c:\Users\ankit\miniconda3\envs\celltypist\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
⛓️ Over-clustering input data with resolution set to 25
🗳️ Majority voting the predictions
✅ Majority voting done!
🔬 Input data has 19038 cells and 27934 genes
🔗 Matching reference genes in the model


AAACCCAAGCATATGA-1    Resident NK
AAACCCACAACGGCTC-1        T cells
AAACCCAGTATTCCGA-1    Resident NK
AAACCCAGTTAAGACA-1        T cells
AAACGAAAGTTGTAGA-1        T cells
Name: typist_liver_majority_voting, dtype: category
Categories (11, object): ['B cells', 'Basophils', 'Circulating NK/NKT', 'Macrophages', ..., 'T cells', 'cDC1s', 'cDC2s', 'pDCs']
AAACCCAAGCATATGA-1    0.999979
AAACCCACAACGGCTC-1    0.993928
AAACCCAGTATTCCGA-1    0.999923
AAACCCAGTTAAGACA-1    0.999995
AAACGAAAGTTGTAGA-1    0.998850
Name: typist_liver_conf_score, dtype: float64
GSE174748



🧬 2485 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 10
🗳️ Majority voting the predictions
✅ Majority voting done!
🔬 Input data has 125790 cells and 37469 genes
🔗 Matching reference genes in the model


AAACCCAAGAGAGTGA-1_GSM5325534_healthy1    Macrophages
AAACCCAAGCGGTAGT-1_GSM5325534_healthy1    Hepatocytes
AAACCCACAAGGGTCA-1_GSM5325534_healthy1    Hepatocytes
AAACCCACACAGCATT-1_GSM5325534_healthy1    Hepatocytes
AAACCCACATCATCTT-1_GSM5325534_healthy1    Hepatocytes
Name: typist_liver_majority_voting, dtype: category
Categories (9, object): ['B cells', 'Cholangiocytes', 'Endothelial cells', 'Fibroblasts', ..., 'Macrophages', 'Plasma cells', 'Resident NK', 'T cells']
AAACCCAAGAGAGTGA-1_GSM5325534_healthy1    1.000000
AAACCCAAGCGGTAGT-1_GSM5325534_healthy1    1.000000
AAACCCACAAGGGTCA-1_GSM5325534_healthy1    0.994462
AAACCCACACAGCATT-1_GSM5325534_healthy1    1.000000
AAACCCACATCATCTT-1_GSM5325534_healthy1    1.000000
Name: typist_liver_conf_score, dtype: float64
GSE185477



...

ND_1_ND_AAACGGGTCATCGATG-1    T cells
ND_1_ND_AAAGATGGTGCATCTA-1    T cells
ND_1_ND_AAAGATGTCACAGTAC-1    T cells
ND_1_ND_AAAGATGTCGCGGATC-1    T cells
ND_1_ND_AAAGCAACAGACAAGC-1    T cells
Name: typist_liver_majority_voting, dtype: category
Categories (2, object): ['Resident NK', 'T cells']
ND_1_ND_AAACGGGTCATCGATG-1    0.040775
ND_1_ND_AAAGATGGTGCATCTA-1    0.958100
ND_1_ND_AAAGATGTCACAGTAC-1    0.768228
ND_1_ND_AAAGATGTCGCGGATC-1    0.997641
ND_1_ND_AAAGCAACAGACAAGC-1    0.999992
Name: typist_liver_conf_score, dtype: float64
GSE192740_liver



🧬 2688 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 15
🗳️ Majority voting the predictions
✅ Majority voting done!
🔬 Input data has 30738 cells and 30704 genes
🔗 Matching reference genes in the model


ABU8_AAACCCAAGGCATTTC-1-7          Hepatocytes
ABU8_AAACCCACAACTGTGT-1-7          Hepatocytes
ABU8_AAACCCATCTTAGCCC-1-7    Endothelial cells
ABU8_AAACGAACAGAAATTG-1-7          Hepatocytes
ABU8_AAACGCTAGGATTTGA-1-7          Hepatocytes
Name: typist_liver_majority_voting, dtype: category
Categories (10, object): ['B cells', 'Cholangiocytes', 'Circulating NK/NKT', 'Endothelial cells', ..., 'Macrophages', 'Plasma cells', 'Resident NK', 'T cells']
ABU8_AAACCCAAGGCATTTC-1-7    0.615399
ABU8_AAACCCACAACTGTGT-1-7    1.000000
ABU8_AAACCCATCTTAGCCC-1-7    1.000000
ABU8_AAACGAACAGAAATTG-1-7    1.000000
ABU8_AAACGCTAGGATTTGA-1-7    0.999974
Name: typist_liver_conf_score, dtype: float64
GSE192740_immune



🧬 2688 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 15
🗳️ Majority voting the predictions
✅ Majority voting done!


CS110_AAACCCACACGGTGTC-1-0    Macrophages
CS110_AAACCCACAGTCAGTT-1-0    Resident NK
CS110_AAACCCACATTGCAAC-1-0        T cells
CS110_AAACCCAGTGTGCTTA-1-0        T cells
CS110_AAACCCAGTGTTCATG-1-0        T cells
Name: typist_liver_majority_voting, dtype: category
Categories (16, object): ['B cells', 'Basophils', 'Cholangiocytes', 'Circulating NK/NKT', ..., 'T cells', 'cDC1s', 'cDC2s', 'pDCs']
CS110_AAACCCACACGGTGTC-1-0    1.000000
CS110_AAACCCACAGTCAGTT-1-0    0.999957
CS110_AAACCCACATTGCAAC-1-0    0.999873
CS110_AAACCCAGTGTGCTTA-1-0    0.999996
CS110_AAACCCAGTGTTCATG-1-0    0.998971
Name: typist_liver_conf_score, dtype: float64


In [21]:
# Save each anndata object in the dictionary to a new h5ad file with the same GSE file name in the directory called 'h5s_common_directory_V2_WIP'

output_dir = BASE / 'h5s_common_directory_V2_WIPAnnotated'
output_dir.mkdir(exist_ok=True)  # Create output directory if it doesn't exist
for name, adata in adata_dict.items():
    output_file = output_dir / f"{name}.h5ad"
    adata.write_h5ad(output_file)
    print(f"Saved {name} to {output_file}")

Saved GSE159977 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE159977.h5ad
Saved GSE174748 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE174748.h5ad
Saved GSE185477 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE185477.h5ad
Saved GSE189600 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE189600.h5ad
Saved GSE190487 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE190487.h5ad
Saved GSE202379 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE202379.h5ad
Saved GSE212837 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE212837.h5ad
Saved GSE270488 to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\GSE270488.h5ad
Saved GSE192740_liver to c:\Users\ankit\Documents\scFM\train_data\h5s_common_directory_V2_WIPAnnotated\G

Checkpoint

In [3]:
# Read in each h5ad file with the Anndata object variable name being it's file name without the extension and store in dictionary
output_dir = BASE / 'h5s_common_directory_V2_WIPAnnotated'
adata_dict = {}
for h5ad_file in output_dir.glob('*.h5ad'):
    adata_name = h5ad_file.stem  # Get file name without extension
    adata_dict[adata_name] = ad.read_h5ad(h5ad_file)  # Read h5ad file and store in dictionary

In [4]:
# 194 351 cells total before filtering for confidence
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

score_col = "typist_liver_conf_score"

# Choose thresholds from 50% to 100%
# Change step to 0.01 if you want finer thresholds
thresholds = np.round(np.arange(0.50, 1.001, 0.05), 2)

# Collect confidence scores from all adata objects
rows = []

for gse_name, adata in adata_dict.items():
    if score_col not in adata.obs.columns:
        print(f"WARNING: {gse_name} does not have {score_col}. Skipping.")
        continue

    scores = pd.to_numeric(adata.obs[score_col], errors="coerce")

    temp = pd.DataFrame({
        "GSE": gse_name,
        "conf_score": scores.values
    })

    rows.append(temp)

liver_conf_df = pd.concat(rows, ignore_index=True)

print(f"Total cells across included objects: {liver_conf_df.shape[0]:,}")
print(f"Cells with non-missing {score_col}: {liver_conf_df['conf_score'].notna().sum():,}")
print(f"Cells with missing {score_col}: {liver_conf_df['conf_score'].isna().sum():,}")

# Overall threshold impact
overall_threshold_summary = []

total_cells = liver_conf_df.shape[0]

for thresh in thresholds:
    kept = (liver_conf_df["conf_score"] >= thresh).sum()
    removed = total_cells - kept

    overall_threshold_summary.append({
        "threshold": thresh,
        "cells_kept": kept,
        "cells_removed": removed,
        "percent_kept": kept / total_cells * 100,
        "percent_removed": removed / total_cells * 100
    })

overall_threshold_summary = pd.DataFrame(overall_threshold_summary)

display(overall_threshold_summary)

Total cells across included objects: 679,904
Cells with non-missing typist_liver_conf_score: 679,904
Cells with missing typist_liver_conf_score: 0


,threshold,cells_kept,cells_removed,percent_kept,percent_removed
0,0.50,598499,81405,88.026986,11.973014
1,0.55,595350,84554,87.563833,12.436167
2,0.60,592090,87814,87.084353,12.915647
3,0.65,588601,91303,86.571192,13.428808
4,0.70,584811,95093,86.013761,13.986239
5,0.75,580603,99301,85.394850,14.605150
6,0.80,575525,104379,84.647980,15.352020
7,0.85,569118,110786,83.705641,16.294359
8,0.90,560402,119502,82.423695,17.576305
9,0.95,546235,133669,80.340019,19.659981
